# Ingest books into `crawler.books_b0`

Parses every document in a folder (pdf, epub, docx, pptx), splits chapters and appends them to the Delta table through Spark Connect.

In [ ]:
import sys
sys.path[:0] = ["../book_tools/book-parser/src", "../book_tools/book-ingest/src"]  # or pip install the packages

INBOX = "/home/user/Documents/chomsky_books"  # folder with the books
SPARK_CONNECT_URL = "sc://localhost:15002"

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote(SPARK_CONNECT_URL).getOrCreate()

## Preview the chapter split (no Spark needed)

In [1]:
from pathlib import Path
import sys
sys.path[:0] = ["../book_tools/book-parser/src", "../book_tools/book-ingest/src"]  # or pip install the packages

from book_parser import default_parser

INBOX = "/home/user/Documents/chomsky_books"  # folder with the books

book = default_parser().parse(next(Path(INBOX).glob("*.epub")))
print(book.title, "|", book.author, "|", book.publication_date, "|", book.split_strategy)
for c in book.chapters:
    print(c.number, c.part, c.name, len(c.text))

Imperial Ambitions | DAVID BARSAMIAN | 2005-01-01 | table_of_contents
1 None INTRODUCTION 2276
2 None 1. IMPERIAL AMBITIONS 20438
3 None 2. COLLATERAL LANGUAGE 29709
4 None 3. REGIME CHANGE 28513
5 None 4. WARS OF AGGRESSION 35953
6 None 5. HISTORY AND MEMORY 29629
7 None 6. THE DOCTRINE OF GOOD INTENTIONS 30389
8 None 7. INTELLECTUAL SELF-DEFENSE 38658
9 None 8. DEMOCRACY AND EDUCATION 16397
10 None 9. ANOTHER WORLD IS POSSIBLE 22404
11 None NOTES 27285
12 None INDEX 11254


## Ingest the folder

Files are moved to `processed/` or `failed/` inside the folder; already-known books (same file hash) are skipped.

In [ ]:
import logging
from book_ingest import BooksRepository, FileArchiver, FolderScanner, IngestJob

logging.basicConfig(level=logging.INFO)
result = IngestJob(
    FolderScanner(INBOX),
    default_parser(),
    BooksRepository(spark),
    FileArchiver(INBOX),
).run()
result

In [ ]:
spark.table("crawler.books_b0").select(
    "filename", "author", "publication_date", "chapter_number", "chapter_name"
).show(30, truncate=60)